# Install (offline Kaggle)

In [1]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [2]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl

Processing /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl


# Reproducibility

In [3]:
import torch, random, numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Paths

In [4]:
TRAIN_SEQ = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv"

VAL_SEQ   = "/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_sequences.csv"
VAL_LBL   = "/kaggle/input/competitions/stanford-rna-3d-folding-2/validation_labels.csv"

TEST_SEQ  = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"

# Dataset

In [5]:
import pandas as pd
from torch.utils.data import Dataset
from Bio.Seq import Seq

NUC_MAP = {'A':0,'U':1,'G':2,'C':3}

def clean_sequence(seq):
    seq = str(Seq(seq.upper()))
    return "".join([s for s in seq if s in NUC_MAP])

def one_hot(seq):
    x = torch.zeros(len(seq),4)
    for i,s in enumerate(seq):
        x[i,NUC_MAP[s]] = 1
    return x


class RNADataset(Dataset):

    def __init__(self, seq_csv, label_csv=None, max_length=1000):

        self.q_df = pd.read_csv(seq_csv)
        self.max_length = max_length
        self.has_labels = label_csv is not None

        if self.has_labels:

            labels = pd.read_csv(label_csv, low_memory=False)
            labels["struct_id"] = labels["ID"].str.split("_").str[0]
            labels["res_idx"] = labels["ID"].str.split("_").str[1].astype(int)

            self.structures = {}

            for sid,g in labels.groupby("struct_id"):

                g = g.sort_values("res_idx")
                coords = torch.tensor(
                    g[["x_1","y_1","z_1"]].values,
                    dtype=torch.float32
                )

                valid = ~torch.isnan(coords).any(dim=1)
                coords = coords[valid]

                if coords.shape[0] > 0:
                    self.structures[sid] = coords

            self.valid_ids = [
                sid for sid in self.q_df["target_id"]
                if sid in self.structures
            ]
        else:
            self.valid_ids = list(self.q_df["target_id"])

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):

        sid = self.valid_ids[idx]
        row = self.q_df[self.q_df["target_id"]==sid].iloc[0]

        seq = clean_sequence(row["sequence"])

        if self.has_labels:

            coords = self.structures[sid]

            L = min(len(seq), coords.shape[0])
            seq = seq[:L]
            coords = coords[:L]

            coords = coords - coords.mean(0,keepdim=True)
            coords = coords / (coords.std()+1e-8)

        else:
            coords=None
            L=len(seq)

        if L>self.max_length:
            seq=seq[:self.max_length]
            if coords is not None:
                coords=coords[:self.max_length]

        x = one_hot(seq)

        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        return sid,x,coords

# Build Datasets

In [6]:
train_dataset = RNADataset(TRAIN_SEQ, TRAIN_LBL)
val_dataset   = RNADataset(VAL_SEQ, VAL_LBL)
test_dataset  = RNADataset(TEST_SEQ, None)

# Graph Builder

In [7]:
from torch_geometric.data import Data

def center_coordinates(coords):
    return coords - coords.mean(dim=0, keepdim=True)

def build_graph(x, coords=None, k=2):

    L = x.size(0)
    edge_index=[]
    edge_attr=[]

    if coords is not None:
        coords=center_coordinates(coords)

    for i in range(L):
        for j in range(max(0,i-k),min(L,i+k+1)):
            if i==j: continue

            edge_index.append([i,j])

            if coords is not None:
                d=torch.norm(coords[i]-coords[j])
                edge_attr.append([d.item()])
            else:
                edge_attr.append([abs(i-j)])

    edge_index=torch.tensor(edge_index).t().contiguous()
    edge_attr=torch.tensor(edge_attr,dtype=torch.float)

    data=Data(x=x,edge_index=edge_index,edge_attr=edge_attr)

    if coords is not None:
        data.pos=coords
        data.y=coords

    return data

# Build Graph Lists

In [8]:
train_graphs=[build_graph(x,c) for _,x,c in train_dataset]
val_graphs=[build_graph(x,c) for _,x,c in val_dataset]

test_graphs=[]
for _,x,_ in test_dataset:
    g=build_graph(x,None)
    g.pos=torch.zeros(g.num_nodes,3)
    test_graphs.append(g)

# DataLoaders

In [9]:
from torch_geometric.loader import DataLoader

train_loader=DataLoader(train_graphs,batch_size=4,shuffle=True)
val_loader=DataLoader(val_graphs,batch_size=4)
test_loader=DataLoader(test_graphs,batch_size=4)

# EGNN Layer

In [10]:
import torch.nn as nn
from torch_geometric.utils import scatter

class SimpleEGNNLayer(nn.Module):

    def __init__(self, hidden, edge_dim=1):
        super().__init__()

        self.edge_mlp=nn.Sequential(
            nn.Linear(hidden*2+edge_dim+1,hidden),
            nn.SiLU(),
            nn.Linear(hidden,hidden)
        )

        self.node_mlp=nn.Sequential(
            nn.Linear(hidden*2,hidden),
            nn.SiLU(),
            nn.Linear(hidden,hidden)
        )

        self.coord_mlp=nn.Sequential(
            nn.Linear(hidden,1),
            nn.Tanh()
        )

    def forward(self,x,pos,edge_index,edge_attr):

        row,col=edge_index

        rel=pos[row]-pos[col]
        dist2=(rel**2).sum(dim=1,keepdim=True)

        m=self.edge_mlp(
            torch.cat([x[row],x[col],edge_attr,dist2],dim=1)
        )

        agg=scatter(m,row,dim=0,dim_size=x.size(0),reduce="mean")
        x=self.node_mlp(torch.cat([x,agg],dim=1))

        delta=scatter(self.coord_mlp(m)*rel,row,
                      dim=0,dim_size=pos.size(0),reduce="mean")

        pos=pos+delta

        return x,pos

# EGNN Model

In [11]:
class EGNNModel(nn.Module):

    def __init__(self,in_dim=5,hidden=64,layers=3):
        super().__init__()

        self.embed=nn.Linear(in_dim,hidden)
        self.layers=nn.ModuleList(
            [SimpleEGNNLayer(hidden) for _ in range(layers)]
        )
        self.out=nn.Linear(hidden,3)

    def forward(self,data):

        x=self.embed(data.x)
        pos=data.pos

        for layer in self.layers:
            x,pos=layer(x,pos,data.edge_index,data.edge_attr)

        return self.out(x)

# Kabsch Alignment

In [18]:
import torch


def kabsch_align(P, Q):
    """
    Differentiable Kabsch alignment
    P, Q : (N,3)
    """

    # center
    Pc = P - P.mean(dim=0, keepdim=True)
    Qc = Q - Q.mean(dim=0, keepdim=True)

    # covariance
    H = Pc.T @ Qc

    # SVD
    U, S, Vt = torch.linalg.svd(H)

    # ---------- SAFE reflection handling ----------
    d = torch.sign(torch.det(Vt.T @ U.T))

    I = torch.eye(3, device=P.device)
    I = I.clone()                # ensure new tensor
    I[-1, -1] = d                # NOT inplace on SVD output

    R = Vt.T @ I @ U.T

    # align
    P_aligned = Pc @ R

    return P_aligned, Qc

# Geometry Loss

In [19]:
import torch.nn.functional as F

def pairwise_dist(x):
    diff=x.unsqueeze(1)-x.unsqueeze(0)
    return torch.sqrt((diff**2).sum(-1)+1e-8)


def structure_loss(pred,target,batch):

    coord_loss=0
    dist_loss=0
    smooth_loss=0

    G=batch.max().item()+1

    for g in range(G):

        mask=(batch==g)

        P,T=pred[mask],target[mask]
        if P.size(0)<3: continue

        P,T=kabsch_align(P,T)

        coord_loss+=F.mse_loss(P,T)

        Dp=pairwise_dist(P)
        Dt=pairwise_dist(T)
        dist_loss+=F.l1_loss(Dp,Dt)

        smooth_loss+=F.mse_loss(
            P[1:]-P[:-1],
            T[1:]-T[:-1]
        )

    coord_loss/=G
    dist_loss/=G
    smooth_loss/=G

    total=coord_loss+0.3*dist_loss+0.1*smooth_loss

    return total

# Training Setup

In [20]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=EGNNModel().to(device)

optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)

# Train / Validate

In [21]:
def train_one_epoch():

    model.train()
    total=0

    for batch in train_loader:

        batch=batch.to(device)
        optimizer.zero_grad()

        pred=model(batch)

        loss=structure_loss(pred,batch.y,batch.batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()

        total+=loss.item()

    return total/len(train_loader)


@torch.no_grad()
def validate():

    model.eval()
    total=0

    for batch in val_loader:

        batch=batch.to(device)
        pred=model(batch)

        loss=structure_loss(pred,batch.y,batch.batch)
        total+=loss.item()

    return total/len(val_loader)

# Training Loop

In [22]:
for epoch in range(5):

    tr=train_one_epoch()
    va=validate()

    print(f"Epoch {epoch} | Train {tr:.4f} | Val {va:.4f}")

Epoch 0 | Train 1.2871 | Val 1.4319
Epoch 1 | Train 1.2642 | Val 1.3711
Epoch 2 | Train 1.2500 | Val 1.4888
Epoch 3 | Train 1.2493 | Val 1.4578
Epoch 4 | Train 1.2482 | Val 1.4219


# Test Inference

In [ ]:
@torch.no_grad()
def run_test():

    model.eval()
    preds=[]

    for batch in test_loader:
        batch=batch.to(device)
        preds.append(model(batch).cpu())

    return torch.cat(preds)

test_predictions=run_test()
print(test_predictions.shape)

# Submission

In [ ]:
IDX2NUC={0:'A',1:'U',2:'G',3:'C'}

def onehot_to_base(x):
    return IDX2NUC[int(x[:4].argmax())]

rows=[]
ptr=0

for sid,x,_ in test_dataset:

    L=x.shape[0]
    coords=test_predictions[ptr:ptr+L]
    ptr+=L

    preds=[coords]+[
        coords+0.02*torch.randn_like(coords)
        for _ in range(4)
    ]

    for i in range(L):

        row={
            "ID":f"{sid}_{i+1}",
            "resname":onehot_to_base(x[i]),
            "resid":i+1
        }

        for k in range(5):
            row[f"x_{k+1}"]=float(preds[k][i,0])
            row[f"y_{k+1}"]=float(preds[k][i,1])
            row[f"z_{k+1}"]=float(preds[k][i,2])

        rows.append(row)

submission=pd.DataFrame(rows)
submission.to_csv("submission.csv",index=False)

print("submission.csv saved")